# Statistical tests for the journal version
## BRISC 2025 - Workflows 1-3 cross-comparison

**Catarina Bota | ESB-UCP**

Replaces the **fabricated** statistics in the first journal draft with real, reproducible numbers:

1. **Friedman test + Nemenyi post-hoc** on per-class F1 across workflows W1, W2, W3
2. **McNemar pairwise tests** between workflow predictions on the test set
3. **Spearman rho** between feature-importance vectors of the seven W3 classifiers
4. **Bootstrap 95% CI** on accuracy for each workflow (if W2 notebook didn't already save them)
5. **Critical difference diagram** (visual Nemenyi)

**Prerequisites (must already be generated and saved to Drive):**
- W1: `workflow1_results/wf1_perclass.csv`, `wf1_predictions.csv` (columns: y_true, y_pred)
- W2: `workflow2_results/w2_perclass.csv`, `w2_per_seed.csv` (with y_true/y_pred from cell 7 output of `workflow2_cnn_train.ipynb`)
- W3: `workflow3_results/cnn_features.npz`, `clf_*.pkl` (output of `workflow3_hybrid_all_classifiers.ipynb`)

**Runtime: ~5 min** (no model training).

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install scikit-posthocs -q

import os, pickle, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare, spearmanr
from statsmodels.stats.contingency_tables import mcnemar
import scikit_posthocs as sp
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

DRIVE_BASE  = '/content/drive/MyDrive/brisc_gui'
OUT_DIR     = f'{DRIVE_BASE}/journal_stats'
os.makedirs(OUT_DIR, exist_ok=True)

WF1_DIR = f'{DRIVE_BASE}/workflow1_results'
WF2_DIR = f'{DRIVE_BASE}/workflow2_results'
WF3_DIR = f'{DRIVE_BASE}/workflow3_results'

CLASSES = ['glioma', 'meningioma', 'no_tumor', 'pituitary']

## 2. Load per-class F1 from all workflows

Each workflow must export a `*_perclass.csv` with columns `class, precision, recall, f1, support`.
If a CSV is missing or has a different schema, this cell prints a clear error so you can regenerate it.

In [ ]:
perclass = {}
for name, path in [
    ('W1', f'{WF1_DIR}/wf1_perclass.csv'),
    ('W2', f'{WF2_DIR}/w2_perclass.csv'),
    ('W3', f'{WF3_DIR}/perclass_best.csv'),
]:
    if not os.path.exists(path):
        print(f'MISSING: {name} -> {path}'); continue
    df = pd.read_csv(path)
    df = df[df['class'].isin(CLASSES)].set_index('class').loc[CLASSES]
    perclass[name] = df['f1'].values
    print(f'  {name} per-class F1: {df["f1"].values}')

assert len(perclass) >= 3, 'Need W1, W2, W3 per-class CSVs to run Friedman.'

## 3. Friedman test + Nemenyi post-hoc

n = 4 classes (small sample) -> Friedman is conservative but valid. Reports test statistic, p-value, and pairwise Nemenyi p-values matrix.

In [ ]:
wf_order = sorted(perclass.keys())
f1_matrix = np.stack([perclass[w] for w in wf_order])  # shape (n_workflows, n_classes)
print('Per-class F1 matrix (rows = workflows, cols = classes):')
print(pd.DataFrame(f1_matrix, index=wf_order, columns=CLASSES).round(2))

# Friedman expects samples as positional args (one per group)
stat, p = friedmanchisquare(*f1_matrix)
print(f'\nFriedman:  chi^2 = {stat:.4f}  p = {p:.4f}')

if p < 0.05:
    print('  -> reject H0 of equivalent performance distributions (alpha=0.05)')
    posthoc = sp.posthoc_nemenyi_friedman(f1_matrix.T)  # transpose: rows are classes
    posthoc.index = wf_order; posthoc.columns = wf_order
    print('\nNemenyi pairwise p-values:')
    print(posthoc.round(4))
    posthoc.to_csv(f'{OUT_DIR}/nemenyi_pairwise.csv')
else:
    print('  -> fail to reject H0; do not run Nemenyi post-hoc')

with open(f'{OUT_DIR}/friedman_result.json', 'w') as f:
    json.dump({'chi2': float(stat), 'p_value': float(p),
               'n_workflows': len(wf_order), 'n_classes': len(CLASSES)}, f, indent=2)

## 4. Critical-difference diagram (visual Nemenyi)

Standard tool from the Demsar (2006) statistical comparison framework. CD = q_alpha * sqrt(k(k+1)/(6N)) where k = workflows, N = classes.

In [ ]:
# Mean ranks across classes (1 = best)
ranks = np.argsort(np.argsort(-f1_matrix, axis=0), axis=0) + 1  # rank within each class
mean_ranks = ranks.mean(axis=1)
print('Mean ranks (1 = best):')
for w, r in zip(wf_order, mean_ranks):
    print(f'  {w}: {r:.2f}')

# Nemenyi critical difference at alpha=0.05 (q values from Demsar 2006 Table 5)
q_alpha = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949}
k = len(wf_order); N = f1_matrix.shape[1]
if k in q_alpha:
    CD = q_alpha[k] * np.sqrt(k*(k+1)/(6*N))
    print(f'\nNemenyi critical difference (alpha=0.05, k={k}, N={N}): CD = {CD:.3f}')

    fig, ax = plt.subplots(figsize=(8, 2.5))
    sorted_idx = np.argsort(mean_ranks)
    sorted_ranks = mean_ranks[sorted_idx]
    sorted_names = [wf_order[i] for i in sorted_idx]
    ax.scatter(sorted_ranks, [1]*k, s=80, color='#1F4E4A', zorder=3)
    for r, n in zip(sorted_ranks, sorted_names):
        ax.annotate(n, (r, 1), textcoords='offset points', xytext=(0, 12),
                     ha='center', fontsize=10, fontweight='bold')
    ax.plot([sorted_ranks[0], sorted_ranks[0]+CD], [0.85, 0.85], color='#C24B4B', lw=3)
    ax.annotate(f'CD = {CD:.3f}',
                ((sorted_ranks[0]*2 + CD)/2, 0.85),
                textcoords='offset points', xytext=(0, -15), ha='center',
                color='#C24B4B', fontsize=10)
    ax.set_xlim([0.5, k+0.5]); ax.set_ylim([0.5, 1.5])
    ax.set_xlabel('Mean rank (lower = better)'); ax.set_yticks([])
    ax.set_title('Critical difference diagram (Nemenyi post-hoc)', fontweight='bold')
    for s in ['top','right','left']: ax.spines[s].set_visible(False)
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/cd_diagram.png', dpi=150, bbox_inches='tight')
    plt.savefig(f'{OUT_DIR}/cd_diagram.pdf', bbox_inches='tight')
    plt.show()

## 5. Load test-set predictions for each workflow

Needed for McNemar pairwise tests. Each workflow notebook should save a CSV with `y_true` and `y_pred` columns. If missing, the cell prints what to add to which notebook.

In [ ]:
preds = {}
for name, path in [
    ('W1', f'{WF1_DIR}/wf1_predictions.csv'),
    ('W2', f'{WF2_DIR}/w2_predictions.csv'),
    ('W3', f'{WF3_DIR}/wf3_predictions.csv'),
]:
    if not os.path.exists(path):
        print(f'MISSING: {name} -> {path}'); continue
    df = pd.read_csv(path)
    preds[name] = (df['y_true'].values, df['y_pred'].values)
    print(f'  {name}: {len(df)} predictions  acc {(df.y_true==df.y_pred).mean()*100:.2f}%')

if 'W2' not in preds and os.path.exists(f'{WF2_DIR}/w2_per_seed.csv'):
    print('\nHint: in workflow2_cnn_train.ipynb, after the training loop, add a cell:')
    print("  pd.DataFrame({'y_true': r0['y_true'], 'y_pred': r0['y_pred']})"
          ".to_csv(f'{OUTPUT_DIR}/w2_predictions.csv', index=False)")

## 6. McNemar pairwise tests

Tests whether two classifiers disagree symmetrically on the test set. The exact (small-sample) version is used because the disagreement counts can be small for the diagonal cells.

In [ ]:
if len(preds) >= 2:
    names = list(preds.keys())
    mc_results = []
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            a, b = names[i], names[j]
            yt_a, yp_a = preds[a]; yt_b, yp_b = preds[b]
            assert (yt_a == yt_b).all(), f'Test labels differ between {a} and {b}'
            n01 = int(((yp_a == yt_a) & (yp_b != yt_b)).sum())
            n10 = int(((yp_a != yt_a) & (yp_b == yt_b)).sum())
            tbl = [[0, n01], [n10, 0]]
            res = mcnemar(tbl, exact=True)
            mc_results.append({
                'pair': f'{a} vs {b}',
                f'{a}_correct_only': n01,
                f'{b}_correct_only': n10,
                'statistic': res.statistic, 'p_value': res.pvalue,
            })
    mc_df = pd.DataFrame(mc_results)
    print(mc_df.to_string(index=False))
    mc_df.to_csv(f'{OUT_DIR}/mcnemar_pairwise.csv', index=False)
else:
    print('Need at least 2 workflows with prediction CSVs.')

## 7. Spearman rho between feature-importance vectors (W3, 7 classifiers)

Tests whether different downstream classifiers exploit the same CNN-feature dimensions.
Uses native `feature_importances_` for tree models and permutation importance for SVM/MLP.

In [ ]:
# Load cached CNN features (needed for permutation importance on SVM/MLP)
data = np.load(f'{WF3_DIR}/cnn_features.npz', allow_pickle=True)
X_test, y_test = data['X_test'], data['y_test']
scaler = StandardScaler().fit(data['X_train'])
X_test_s = scaler.transform(X_test)
print(f'Test features: {X_test_s.shape}')

# Map of {pretty name -> pkl file basename}
clf_files = {
    'XGBoost':         'clf_XGBoost.pkl',
    'LightGBM':        'clf_LightGBM.pkl',
    'RandomForest':    'clf_RandomForest.pkl',
    'GradientBoosting':'clf_GradientBoosting.pkl',
    'CatBoost':        'clf_CatBoost.pkl',
    'SVM_RBF':         'clf_SVM_RBF.pkl',
    'MLP':             'clf_MLP.pkl',
}

importances = {}
for name, fn in clf_files.items():
    path = f'{WF3_DIR}/{fn}'
    if not os.path.exists(path):
        print(f'  SKIP {name}: {path} missing'); continue
    with open(path, 'rb') as f:
        d = pickle.load(f)
    clf = d['clf']
    if hasattr(clf, 'feature_importances_'):
        imps = clf.feature_importances_
        kind = 'native'
    else:
        # Permutation importance on a 300-img subset for speed
        result = permutation_importance(
            clf, X_test_s[:300], y_test[:300],
            n_repeats=5, random_state=42, n_jobs=-1)
        imps = result.importances_mean
        kind = 'permutation'
    importances[name] = imps / (imps.sum() + 1e-12)  # L1-normalise
    print(f'  {name}: importance vector computed ({kind})')

In [ ]:
names = sorted(importances.keys())
K = len(names)
rho_mat = np.zeros((K, K))
p_mat   = np.zeros((K, K))
for i in range(K):
    for j in range(K):
        if i == j:
            rho_mat[i,j] = 1.0
        else:
            r, p = spearmanr(importances[names[i]], importances[names[j]])
            rho_mat[i,j] = r; p_mat[i,j] = p

rho_df = pd.DataFrame(rho_mat, index=names, columns=names).round(3)
print('Spearman rho matrix:')
print(rho_df)
rho_df.to_csv(f'{OUT_DIR}/spearman_rho_matrix.csv')

# Heatmap
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(rho_mat, cmap='YlGnBu', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(K)); ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_yticks(range(K)); ax.set_yticklabels(names)
for i in range(K):
    for j in range(K):
        ax.text(j, i, f'{rho_mat[i,j]:.2f}', ha='center', va='center',
                color='white' if rho_mat[i,j] > 0.5 else 'black', fontsize=9)
fig.colorbar(im, ax=ax)
ax.set_title('Cross-classifier feature-importance Spearman rho',
              fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/spearman_rho_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Headline number for the journal: mean off-diagonal rho
off = rho_mat[np.triu_indices(K, k=1)]
print(f'\nMean off-diagonal rho: {off.mean():.3f}  (min {off.min():.3f}, max {off.max():.3f})')

## 8. (Optional) Per-workflow bootstrap CI on accuracy

If `w2_per_seed.csv` already has `acc_ci_lo`/`acc_ci_hi` they are used directly. Otherwise this cell computes them from `*_predictions.csv`.

In [ ]:
def bootstrap_acc_ci(y_true, y_pred, n_iter=1000, alpha=0.05, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    accs = [(y_true[rng.randint(0,n,n)] == y_pred[rng.randint(0,n,n)]).mean()*100
            for _ in range(n_iter)]
    return float(np.percentile(accs, 100*alpha/2)), \
           float(np.percentile(accs, 100*(1-alpha/2)))

ci_rows = []
for name, (yt, yp) in preds.items():
    acc = (yt == yp).mean()*100
    lo, hi = bootstrap_acc_ci(yt, yp)
    ci_rows.append({'workflow': name, 'accuracy': round(acc,2),
                    'ci95_lo': round(lo,2), 'ci95_hi': round(hi,2)})
ci_df = pd.DataFrame(ci_rows)
print(ci_df.to_string(index=False))
ci_df.to_csv(f'{OUT_DIR}/bootstrap_ci_per_workflow.csv', index=False)

## 9. Headline numbers for the journal

These are the **only** statistical claims you should put in the article. Replace any leftover hand-written number from the earlier draft.

In [ ]:
report_lines = [
    '=' * 60,
    'STATISTICAL CLAIMS YOU CAN MAKE IN THE PAPER',
    '=' * 60,
    f'\nFriedman: chi^2 = {stat:.3f}, p = {p:.4f}  (n_workflows={len(wf_order)}, n_classes={len(CLASSES)})',
]
if p < 0.05 and k in q_alpha:
    report_lines.append(f'Nemenyi CD (alpha=0.05): {CD:.3f}')
    report_lines.append(f'Mean ranks: ' + ', '.join(f'{w} {r:.2f}' for w, r in zip(wf_order, mean_ranks)))
if len(preds) >= 2 and len(mc_results) > 0:
    report_lines.append('\nMcNemar pairwise:')
    for r in mc_results:
        report_lines.append(f"  {r['pair']}: p = {r['p_value']:.4f}")
if len(importances) >= 2:
    report_lines.append(f'\nMean off-diagonal cross-classifier Spearman rho: {off.mean():.3f}')
if len(ci_rows) > 0:
    report_lines.append('\nBootstrap 95% CI:')
    for r in ci_rows:
        report_lines.append(f"  {r['workflow']}: {r['accuracy']:.2f}% [{r['ci95_lo']:.2f}, {r['ci95_hi']:.2f}]")
report_lines.append('\n' + '=' * 60)
report = '\n'.join(report_lines)
print(report)
with open(f'{OUT_DIR}/journal_headline_stats.txt', 'w') as f:
    f.write(report)
print(f'\nSaved to {OUT_DIR}/journal_headline_stats.txt')

## What this notebook replaces in the journal draft

Every number in this notebook is **measured, not invented**. After running, manually update `artigo_journal.tex`:
- Replace `[95.7, 97.2]` etc with the values from `bootstrap_ci_per_workflow.csv`
- Replace `chi^2`/`p_value` with the Friedman result
- Replace `CD = 1.05` with the real Nemenyi CD
- Replace `rho = 0.78` with the mean off-diagonal Spearman rho
- Add the McNemar p-values to a small in-text or footnote table

If any cell errors out with a missing-file message, the fix is in the workflow notebook that produces that CSV --- not here.